### **The End-to-End A/B Testing Framework (Memo & Checklist)**

- Name: Mingjie Wei

#### **Phase 1: Experiment Design & Goal Definition (Before - Design)**

**1. Define a Specific & Testable Hypothesis**

- Step: Clearly articulate what is changing (Treatment), the expected outcome, and the bottom line (what must not get worse).

- Example: “Changing the homepage hero banner to include a new CTA will increase the 7-day Revenue Per User (RPU) without increasing the customer support ticket rate.”

**2. Determine Unit of Observation & Randomization**

- Step: Choose the randomization unit (e.g., User ID, Cookie, Session) and use a stable assignment method (like hashing) to ensure the same user stays in the same variant.

- Check (SUTVA): Ensure the Stable Unit Treatment Value Assumption holds (individuals do not interfere with each other). If interference exists (e.g., social networks, two-sided marketplaces like Uber/DoorDash), consider cluster randomization or time-switch designs.

- Example: Using User ID hash for a 50/50 split ensures a user logging in from different browsers sees the same treatment.

**3. Define Scope & Audience**

- Step: Specify the surfaces (PC/Mobile, iOS/Android, specific countries) and set clear Include/Exclude rules.

- Example: Include all logged-in users; Exclude internal employee IPs and known bot traffic.

**4. Build the Metrics Framework**

- Step: Define the three tiers of metrics based on the hypothesis.

    - OEC (Overall Evaluation Criterion): Must be Timely, Sensitive, Measurable, and Attributable.

    - Guardrails: Protective (ensure no harm is done) and Trust (validate the experiment's integrity).

    - Diagnostic: Secondary metrics to explain why the OEC moved.

- Example: OEC: 7-day RPU. Guardrails: Page latency (Protective), Sample ratio (Trust). Diagnostic: CTA Click-Through Rate (CTR).

#### **Phase 2: Data & Statistical Preparation (Before - Data)**

**1. Determine Minimal Detectable Effect (MDE)**

- Step: Align with business stakeholders on the smallest effect size that would justify launching the new feature (Practical Significance).

- Example: We need at least a +2% lift in RPU to cover the engineering costs of maintaining the new feature.

**2. Power Analysis & Sample Size Calculation**

- Step: Calculate the required sample size per variant using Baseline conversion, MDE, Significance Level ($\alpha$), and Statistical Power ($1-\beta$).

- Check: If the test is expensive or risky, or if you plan to do heavy sub-population analysis, increase your Power and lower your $\alpha$.

**3. Determine Run Time & Duration**

- Step: Divide the required sample size by the expected daily eligible traffic to get the duration in days.

- Check (Seasonality): Ensure the duration covers full business cycles (e.g., at least 7 or 14 days to capture weekend vs. weekday behavior) and allows time for Novelty/Primacy effects to wear off.

#### **Phase 3: Launch & Execution (Experiment Launch)**

**1. Run A/A Validation**

- Step: Run an A/A test (both groups get the Control experience) for a few days before deploying the Treatment.

- Check: Verify that randomization works and there is no inherent bias or significant difference in baseline metrics between the two groups.

**2. Launch, Monitor, and Wait (No Peeking!)**

- Step: Start the A/B test and let it run for the pre-calculated duration. Do not stop the experiment early just because a p-value becomes significant. 

- Check (Kill Switch): Monitor only the protective guardrails daily. Have a pre-defined threshold to abort the experiment if something breaks.

- Example: If crash rates spike by more than 5% in the Treatment group on Day 2, roll back immediately.

#### **Phase 4: Post-Experiment Validation & Interpretation (Analysis)**

**1. Internal Validity Checks**

- Step: Validate data quality before reading the results.

- Check: 

    - SRM (Sample Ratio Mismatch): Is the traffic actually split 50/50? If not, investigate logging bugs.

    - Covariate Balance: Are user characteristics (e.g., OS distribution) equal across variants?

**2. Interpret Results (Statistical vs. Practical Significance)**

- Step: Evaluate the OEC and Standard Errors.

- Decision Matrix:

    - Statistically & Practically Significant: Great! The feature works.

    - Stat. Significant but NOT Practically Significant: You have a massive sample size detecting a tiny change. It might not be worth the cost to launch.

    - Practically Significant but NOT Stat. Significant: Underpowered experiment. Consider running it longer if safe to do so.

    - Neither: Check standard errors. If small, it's a true null effect. If huge, the experiment was poorly designed.

**3. Segmentation & Heterogeneous Effects**

- Step: Slice the data by sub-populations to see if the treatment impacted specific groups differently.

- Check: Apply Multi-test Corrections (e.g., Benjamini-Hochberg for FDR, or Bonferroni) to avoid false positives from slicing the data too many ways.

- Example: Checking if the uplift was entirely driven by iOS users while Android users saw a drop.

#### **Phase 5: External Validity & Reporting (External Validity)**

**1. Assess External Validity**

- Step: Consider if the results will generalize to the broader population over time.

- Check: Did external factors (holidays, competitor launches) skew the results?

**2. Standardized Reporting**

- Step: Create a concise 1-page summary for stakeholders.

- Structure: Background $\rightarrow$ Setup $\rightarrow$ Results $\rightarrow$ Risks/Guardrails $\rightarrow$ Final Decision (Ship / No-Ship / Iterate) $\rightarrow$ Appendix (SRM, A/A checks, definitions).

### **Appendix: Essential A/B Testing Formulas**

**1. Sample Size Calculation (Continuous Metric)**

To estimate the sample size $n$ per variant:

$$n = \frac{(Z_{1-\alpha/2} + Z_{1-\beta})^2 \cdot (\sigma_1^2 + \sigma_2^2)}{(\mu_1 - \mu_2)^2}$$

(Where $\mu_1 - \mu_2$ is the MDE, $\sigma^2$ is the variance, $\alpha$ is the significance level, and $1-\beta$ is the statistical power).

**2. Sample Size Calculation (Rule of Thumb / Quick Math)**

For a standard setup ($\alpha = 0.05$, Power = $80\%$), the formula roughly simplifies to:

$$n \approx \frac{16 \sigma^2}{\Delta^2}$$

(Where $\Delta$ is the absolute MDE).

**3. Standard Error (Proportions/Conversion Rates)**

For binary metrics (like Click-Through Rate), the standard error is:

$$SE = \sqrt{\frac{p(1-p)}{n}}$$

**4. Two-Sample Z-Test Statistic**

Used to compare the means of two independent groups:

$$Z = \frac{(\bar{X}_1 - \bar{X}_2)}{\sqrt{\frac{\sigma_1^2}{n_1} + \frac{\sigma_2^2}{n_2}}}$$

**5. Sample Ratio Mismatch (SRM) Check**

Used to test if the observed traffic split matches the expected split (e.g., 50/50), using a Chi-Square goodness-of-fit test:

$$\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}$$

(Where $O_i$ is the Observed traffic in variant $i$, and $E_i$ is the Expected traffic).